# OpenDDE — all-atom biomolecular co-folding structure prediction

[OpenDDE](https://github.com/aurekaresearch/OpenDDE) is an all-atom co-folding model that jointly predicts the 3D structures of proteins, nucleic acids (DNA and RNA), small-molecule ligands, and their complexes. This notebook demonstrates `run_opendde` end to end: building inputs, configuring the model, running a prediction, inspecting confidence metrics, and exporting the result.

> **Note:** The prediction cells require a GPU and provisioned OpenDDE model weights. They will not run on a CPU-only or un-provisioned environment.

In [ ]:
from pathlib import Path

from proto_tools.utils.notebook_docs import display_api_reference
from proto_tools.tools.structure_prediction.opendde import run_opendde, OpenDDEInput, OpenDDEConfig
from proto_tools import Chain, Complex

## API reference

The input, configuration, and output schemas for `run_opendde` are rendered directly from the tool definition below.

In [ ]:
# Input schema
display_api_reference("opendde-prediction", "input", "run_opendde")

In [ ]:
# Configuration schema
display_api_reference("opendde-prediction", "config", "run_opendde")

In [ ]:
# Output schema
display_api_reference("opendde-prediction", "output", "run_opendde")

## Basic usage

We fold Trp-cage (TC5b), a real 20-residue mini-protein that adopts a compact folded structure, making it a fast target for a first prediction. We keep the configuration minimal by disabling the MSA homology search (`use_msa=False`) for single-sequence prediction.

> **GPU required:** the cell below runs the OpenDDE model and needs a GPU plus provisioned OpenDDE weights.

In [ ]:
# Trp-cage TC5b — a real 20-residue mini-protein
trpcage_sequence = "NLYIQWLKDGGPSSGRPPPS"

inputs = OpenDDEInput(
    complexes=[Complex(chains=[Chain(sequence=trpcage_sequence, entity_type="protein")])]
)

# Minimal, fast configuration: single-sequence mode (no MSA search)
config = OpenDDEConfig(
    use_msa=False,
    device="cuda",  # requires a GPU; OpenDDE weights must be provisioned
)

result = run_opendde(inputs, config)

structure = result.structures[0]
print(f"Protein length: {len(trpcage_sequence)} residues")
print(f"Average pLDDT:  {structure.metrics['avg_plddt']:.1f}")
print(f"pTM score:      {structure.metrics['ptm']:.3f}")

## Advanced usage: multi-chain complex

OpenDDE can fold multi-chain complexes and report an interface confidence (`iptm`) between chains. Here we fold human insulin — a real two-chain complex whose A and B chains associate through disulfide bonds — and tune the sampling configuration: `num_samples=2` keeps the best of two diffusion samples (by ranking score), while `num_steps=100` trades a little accuracy for speed.

> **GPU required:** the cell below runs the OpenDDE model and needs a GPU plus provisioned OpenDDE weights.

In [ ]:
# Human insulin — a real two-chain protein complex (A chain + B chain)
insulin_a_chain = "GIVEQCCTSICSLYQLENYCN"
insulin_b_chain = "FVNQHLCGSHLVEALYLVCGERGFFYTPKT"

insulin_complex = Complex(
    chains=[
        Chain(sequence=insulin_a_chain, entity_type="protein"),
        Chain(sequence=insulin_b_chain, entity_type="protein"),
    ]
)
inputs = OpenDDEInput(complexes=[insulin_complex])

config = OpenDDEConfig(
    model_name="opendde_v1",
    num_samples=2,  # keep the best of 2 diffusion samples by ranking score
    num_steps=100,  # fewer denoising steps for a faster demo
    use_msa=False,
    device="cuda",  # requires a GPU; OpenDDE weights must be provisioned
)

result = run_opendde(inputs, config)

structure = result.structures[0]
print(f"Chains:         {len(insulin_complex.chains)}")
print(f"Average pLDDT:  {structure.metrics['avg_plddt']:.1f}")
print(f"pTM score:      {structure.metrics['ptm']:.3f}")
print(f"ipTM score:     {structure.metrics['iptm']:.3f}")  # interface confidence between chains

## Export results

Predicted structures export to mmCIF or PDB for downstream analysis in tools such as PyMOL, ChimeraX, or VMD. The B-factor column carries per-residue pLDDT confidence scores.

In [ ]:
import tempfile

export_dir = Path(tempfile.mkdtemp())

# mmCIF (B-factor column carries per-residue pLDDT)
result.export(name="insulin_complex", export_path=export_dir, file_format="cif")

# PDB for tools like PyMOL / ChimeraX
result.export(name="insulin_complex", export_path=export_dir, file_format="pdb")

print(f"Structures exported to {export_dir}")
print(f"  {export_dir / 'insulin_complex.cif'}")
print(f"  {export_dir / 'insulin_complex.pdb'}")